# 02 · Data Cleaning & Preparation

**Goal:** turn the raw Olist tables into clean, analysis-ready tables that can be loaded into PostgreSQL
with primary and foreign keys.

Decisions are based on the findings of `01_load.ipynb`:

| Issue | Decision |
| --- | --- |
| Dates stored as text | convert to `datetime` |
| ~3% of orders not delivered | keep `delivered` orders with a delivery date |
| Several reviews per order | keep the latest review per order |
| Products without category | label as `unknown` |
| 2 untranslated categories | add English names manually |
| Payments ≠ items total in some orders | use **order items** as the revenue source |

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

orders = pd.read_csv(RAW / "olist_orders_dataset.csv")
items = pd.read_csv(RAW / "olist_order_items_dataset.csv")
payments = pd.read_csv(RAW / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(RAW / "olist_order_reviews_dataset.csv")
customers = pd.read_csv(RAW / "olist_customers_dataset.csv")
products = pd.read_csv(RAW / "olist_products_dataset.csv")
translation = pd.read_csv(RAW / "product_category_name_translation.csv")

log = []  # keeps a record of every cleaning step


def record(step, before, after):
    log.append({"step": step, "rows_before": before, "rows_after": after, "removed": before - after})

## 1. Orders

### 1.1 Convert dates

In [2]:
DATE_COLS = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for col in DATE_COLS:
    orders[col] = pd.to_datetime(orders[col])

orders[DATE_COLS].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

### 1.2 Keep delivered orders

In [3]:
n = len(orders)
orders = orders[orders["order_status"] == "delivered"]
record("orders: keep status = delivered", n, len(orders))

print("Delivered orders with missing dates:")
orders[DATE_COLS].isna().sum()

Delivered orders with missing dates:


order_purchase_timestamp          0
order_approved_at                14
order_delivered_carrier_date      2
order_delivered_customer_date     8
order_estimated_delivery_date     0
dtype: int64

In [4]:
n = len(orders)
orders = orders.dropna(subset=["order_delivered_customer_date"])
record("orders: drop delivered without delivery date", n, len(orders))
print(f"Removed {n - len(orders)} orders marked delivered but without a delivery date")

Removed 8 orders marked delivered but without a delivery date


The remaining missing `order_approved_at` / `order_delivered_carrier_date` values (16 rows) are kept:
these columns are not used in the analysis, and the orders themselves are valid.

### 1.3 Delivery features

In [5]:
orders["delivery_days"] = (
    orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]
).dt.days

orders["delay_days"] = (
    orders["order_delivered_customer_date"].dt.normalize() - orders["order_estimated_delivery_date"]
).dt.days

orders["is_late"] = orders["delay_days"] > 0

orders[["delivery_days", "delay_days"]].describe().T

,count,mean,std,min,25%,50%,75%,max
delivery_days,"96,470.00",12.09,9.55,0.00,6.00,10.00,15.00,209.00
delay_days,"96,470.00",-11.88,10.18,-147.00,-17.00,-12.00,-7.00,188.00


In [6]:
print(f"Late deliveries: {orders['is_late'].mean():.1%}")
print(f"Negative delivery time: {(orders['delivery_days'] < 0).sum()}")

Late deliveries: 6.8%
Negative delivery time: 0


- Median delivery takes **10 days**; there are no negative durations.
- `delay_days > 0` means the parcel arrived after the date promised to the customer.
- A few extreme values (up to 200+ days) are real but rare; they stay in the data and are
  handled with medians and buckets in the analysis instead of being deleted.

### 1.4 Analysis period

In [7]:
orders["order_purchase_timestamp"].dt.to_period("M").value_counts().sort_index().head(5)

order_purchase_timestamp
2016-09       1
2016-10     265
2016-12       1
2017-01     750
2017-02    1653
Freq: M, Name: count, dtype: int64

2016 has only a few hundred orders with gaps (no orders in Nov 2016), so trend analysis will use
**Jan 2017 – Aug 2018** (20 full months). The 2016 orders stay in the tables and are filtered out in SQL
where needed, so customer histories remain complete.

## 2. Order items

In [8]:
items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"])

n = len(items)
items = items[items["order_id"].isin(orders["order_id"])]
record("order_items: keep items of clean orders", n, len(items))

print("Non-positive prices:", (items["price"] <= 0).sum())
print("Negative freight:   ", (items["freight_value"] < 0).sum())
items[["price", "freight_value"]].describe().T

Non-positive prices: 0
Negative freight:    0


,count,mean,std,min,25%,50%,75%,max
price,"110,189.00",119.98,182.30,0.85,39.90,74.90,134.17,"6,735.00"
freight_value,"110,189.00",19.95,15.70,0.00,13.08,16.26,21.15,409.68


Prices and freight are all valid. Each row is one unit of a product inside an order.

## 3. Payments

In [9]:
n = len(payments)
payments = payments[payments["order_id"].isin(orders["order_id"])]
record("order_payments: keep payments of clean orders", n, len(payments))

payments["payment_type"].value_counts()

payment_type
credit_card    74579
boleto         19191
voucher         5493
debit_card      1485
Name: count, dtype: int64

In [10]:
print("Zero-value payments:", (payments["payment_value"] == 0).sum())
print("Delivered orders without payment:", (~orders["order_id"].isin(payments["order_id"])).sum())

Zero-value payments: 4


Delivered orders without payment: 1


Zero-value payments are vouchers covering part of an order, so they are kept.
The payments table is used for **payment method analysis** only.

### 3.1 Which table defines revenue?

In [11]:
items_total = items.groupby("order_id")[["price", "freight_value"]].sum().sum(axis=1)
payments_total = payments.groupby("order_id")["payment_value"].sum()

check = pd.concat({"items": items_total, "payments": payments_total}, axis=1).dropna()
check["diff"] = check["payments"] - check["items"]
mismatch = check[check["diff"].abs() > 1]

print(f"Orders where payments differ from items by > 1 BRL: {len(mismatch)} ({len(mismatch) / len(check):.2%})")
print(f"Total items value:    {check['items'].sum():,.0f} BRL")
print(f"Total payments value: {check['payments'].sum():,.0f} BRL")

Orders where payments differ from items by > 1 BRL: 246 (0.26%)
Total items value:    15,418,251 BRL
Total payments value: 15,421,083 BRL


The difference is small (interest on installments, vouchers) and affects only 0.3% of orders.
**Revenue is defined from order items:** `price + freight_value`. Items exist for every order and can be
split by product category, while payments cannot.

## 4. Reviews

In [12]:
for col in ["review_creation_date", "review_answer_timestamp"]:
    reviews[col] = pd.to_datetime(reviews[col])

n = len(reviews)
reviews = reviews[reviews["order_id"].isin(orders["order_id"])]
record("order_reviews: keep reviews of clean orders", n, len(reviews))

n = len(reviews)
reviews = (
    reviews.sort_values("review_answer_timestamp")
    .drop_duplicates(subset="order_id", keep="last")
    [["order_id", "review_id", "review_score", "review_creation_date", "review_answer_timestamp"]]
)
record("order_reviews: keep latest review per order", n, len(reviews))

print("Orders without a review:", (~orders["order_id"].isin(reviews["order_id"])).sum())
reviews["review_score"].value_counts().sort_index()

Orders without a review: 646


review_score
1     9351
2     2921
3     7916
4    18888
5    56748
Name: count, dtype: int64

- After deduplication `order_id` is unique and becomes the primary key of the reviews table.
- Comment title and text are dropped: this project analyses scores, not text.
- ~650 orders have no review. They stay in the data and are simply excluded from review-score metrics.

## 5. Products & categories

In [13]:
manual = pd.DataFrame(
    {
        "product_category_name": ["pc_gamer", "portateis_cozinha_e_preparadores_de_alimentos"],
        "product_category_name_english": ["pc_gamer", "portable_kitchen_and_food_preparers"],
    }
)
translation = pd.concat([translation, manual], ignore_index=True)

products = products.merge(translation, on="product_category_name", how="left")
products["product_category_name"] = products["product_category_name"].fillna("unknown")
products["category"] = products["product_category_name_english"].fillna("unknown")

products = products.rename(
    columns={"product_name_lenght": "product_name_length", "product_description_lenght": "product_description_length"}
)
products = products[
    ["product_id", "category", "product_category_name", "product_photos_qty",
     "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]
]

n = len(products)
products = products[products["product_id"].isin(items["product_id"])]
record("products: keep products sold in clean orders", n, len(products))

print("Categories:", products["category"].nunique())
print("Products without category:", (products["category"] == "unknown").sum())

Categories: 74
Products without category: 584


Two original column names contain a typo (`lenght`). Those columns are not needed for the analysis,
so they are dropped; the remaining columns use clean `snake_case` names.

## 6. Customers

In [14]:
n = len(customers)
customers = customers[customers["customer_id"].isin(orders["customer_id"])]
record("customers: keep customers of clean orders", n, len(customers))

customers["customer_city"] = customers["customer_city"].str.strip().str.title()

print(f"customer_id (one per order): {customers['customer_id'].nunique():,}")
print(f"customer_unique_id (person): {customers['customer_unique_id'].nunique():,}")

customer_id (one per order): 96,470
customer_unique_id (person): 93,350


In [15]:
orders_per_person = (
    orders.merge(customers, on="customer_id")
    .groupby("customer_unique_id")["order_id"]
    .nunique()
)
print(f"Customers with 2+ orders: {(orders_per_person > 1).mean():.1%}")
orders_per_person.value_counts().sort_index().head(6)

Customers with 2+ orders: 3.0%


order_id
1    90549
2     2573
3      181
4       28
5        9
6        5
Name: count, dtype: int64

Only about **3% of customers** placed more than one order. This is the first important business
insight: Olist is mostly a one-time purchase platform, so retention is a key growth lever
(analysed in detail in `04_metrics.ipynb`).

## 7. Final checks

In [16]:
tables = {
    "orders": (orders, ["order_id"]),
    "order_items": (items, ["order_id", "order_item_id"]),
    "order_payments": (payments, ["order_id", "payment_sequential"]),
    "order_reviews": (reviews, ["order_id"]),
    "customers": (customers, ["customer_id"]),
    "products": (products, ["product_id"]),
}

pd.DataFrame(
    {
        "rows": {n: len(df) for n, (df, _) in tables.items()},
        "primary_key_unique": {n: not df.duplicated(subset=k).any() for n, (df, k) in tables.items()},
        "key_has_nulls": {n: df[k].isna().any().any() for n, (df, k) in tables.items()},
    }
)

,rows,primary_key_unique,key_has_nulls
orders,96470,True,False
order_items,110189,True,False
order_payments,100748,True,False
order_reviews,95824,True,False
customers,96470,True,False
products,32214,True,False


In [17]:
fk = {
    "orders → customers": orders["customer_id"].isin(customers["customer_id"]).all(),
    "order_items → orders": items["order_id"].isin(orders["order_id"]).all(),
    "order_items → products": items["product_id"].isin(products["product_id"]).all(),
    "order_payments → orders": payments["order_id"].isin(orders["order_id"]).all(),
    "order_reviews → orders": reviews["order_id"].isin(orders["order_id"]).all(),
}
pd.Series(fk, name="all keys found").to_frame()

,all keys found
orders → customers,True
order_items → orders,True
order_items → products,True
order_payments → orders,True
order_reviews → orders,True


### Cleaning log

In [18]:
pd.DataFrame(log)

,step,rows_before,rows_after,removed
0,orders: keep status = delivered,99441,96478,2963
1,orders: drop delivered without delivery date,96478,96470,8
2,order_items: keep items of clean orders,112650,110189,2461
3,order_payments: keep payments of clean orders,103886,100748,3138
4,order_reviews: keep reviews of clean orders,99224,96353,2871
5,order_reviews: keep latest review per order,96353,95824,529
6,products: keep products sold in clean orders,32951,32214,737
7,customers: keep customers of clean orders,99441,96470,2971


## 8. Save

In [19]:
ORDER_COLS = ["order_id", "customer_id", "order_status"] + DATE_COLS + ["delivery_days", "delay_days", "is_late"]

outputs = {
    "orders": orders[ORDER_COLS],
    "order_items": items,
    "order_payments": payments,
    "order_reviews": reviews,
    "customers": customers,
    "products": products,
}

for name, df in outputs.items():
    df.to_csv(PROCESSED / f"{name}.csv", index=False)
    print(f"{name + '.csv':<22} {len(df):>8,} rows")

orders.csv               96,470 rows


order_items.csv         110,189 rows
order_payments.csv      100,748 rows


order_reviews.csv        95,824 rows
customers.csv            96,470 rows


products.csv             32,214 rows


## Summary

- **96,470 delivered orders** from **93,350 customers** are ready for analysis.
- Every table has a unique primary key and all foreign keys match, so the data can be loaded into
  PostgreSQL with constraints.
- Key definitions fixed here and used in every later step:
  - **revenue** = `price + freight_value` from order items;
  - **customer** = `customer_unique_id`;
  - **trend period** = Jan 2017 – Aug 2018;
  - **late delivery** = delivered after the estimated date.
- First insight: only ~3% of customers buy again.

Next step → `src/build_db.py` loads these files into PostgreSQL, then `03_sql_analysis.ipynb`.